## **Classification Model for Cats and dogs** :

First of all mount your drive with colab

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [19]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("bhavikjikadara/dog-and-cat-classification-dataset")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'dog-and-cat-classification-dataset' dataset.
Path to dataset files: /kaggle/input/dog-and-cat-classification-dataset


## Install all required libraries:

First of all we need to import all required libraries.We will use all important libraries for training the model

In [20]:
!pip install split-folders



In [21]:

import tensorflow as tf
import numpy as np
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.optimizers import Adam
import splitfolders
import matplotlib.pyplot as plt

## Splits the data into three parts:

First we need to split ur data into three parts
`Train data`
`Validation data`
`Test data`

As above we use `splitfolders` python library for spliting of data

In [22]:
dataset_path = '/content/drive/MyDrive/PetImages'
path = os.path.join(path, 'PetImages')
splitfolders.ratio(
    path,
    output = '/content/dataset_split_new',
    seed = 1337,
    ratio = (0.70 , 0.15 , 0.15)
)

Copying files: 24998 files [05:52, 70.83 files/s]


### Visualizing Our Data Splits



```text
dataset_split_new/
│
├── train/                 
│   ├── cats/
│   └── dogs/
│
├── val/                   
│   ├── cats/
│   └── dogs/
│
└── test/                 
    ├── cats/
    └── dogs/


Because we copy data for some reason like sync problem,internet connection some data may be crupted.So we need to remove them so our model should be trained

In [23]:
import os
from PIL import Image

folder_path = '/content/dataset_split_new'
removed_count = 0

print("Starting strict image cleanup...")

for root, dirs, files in os.walk(folder_path):
    for file in files:
        file_path = os.path.join(root, file)

        if not file.lower().endswith(('.png', '.jpg', '.jpeg')):
            os.remove(file_path)
            removed_count += 1
            continue

        try:
            img = Image.open(file_path)
            img.load()
        except:
            os.remove(file_path)
            removed_count += 1

print(f"Cleanup complete! Removed {removed_count} corrupted or invalid files.")


Starting strict image cleanup...


/usr/local/lib/python3.12/dist-packages/PIL/TiffImagePlugin.py:950: UserWarning: Truncated File Read
  warnings.warn(str(msg))


Cleanup complete! Removed 0 corrupted or invalid files.


## Making ImageDataGenerator:

First we will use a Data augmentation technique to changes images,apply different effect like flip,rotate and other

In [24]:
train_datagen = ImageDataGenerator(
    rescale=1.0/255.0,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True
)

Now we need to give path of our datasets.So we will use `ImageDataGenertaor`.
It is used for normlization of data

In [25]:


datagen = ImageDataGenerator(
    rescale = 1.0/255.0
)

# **Making Data generators:**

## Train_data generator:
Now we will make train data genertor for model

In [26]:
train_data = train_datagen.flow_from_directory(
    directory = '/content/dataset_split_new/train',
    class_mode = 'binary',
    batch_size = 32,
    target_size = (150,150),
    shuffle = True
)

Found 17498 images belonging to 3 classes.


# Test_data generator:
Now we will make test data genertor for model

In [27]:
test_data = datagen.flow_from_directory(
    directory = '/content/dataset_split_new/test',
    class_mode = 'binary',
    batch_size = 32,
    target_size = (150,150),
    shuffle = False
)

Found 3752 images belonging to 3 classes.


## Val_data generator:
Now we will make validating data genertor for model

In [28]:
val_data = datagen.flow_from_directory(
    directory = '/content/dataset_split_new/val',
    class_mode = 'binary',
    batch_size = 32,
    target_size = (150,150),
    shuffle = False
)

Found 3748 images belonging to 3 classes.


# **Making neural network:**

Now we will make a neural network that will learn from these images.We will use Sequential model for this

In [29]:
model = Sequential(
    [
        Conv2D(32 , (3,3), activation='relu',input_shape = (150,150,3)),
        MaxPooling2D(2,2),

        Conv2D(64 , (3,3),activation='relu'),
        MaxPooling2D(2,2),

        Conv2D(128 , (3,3), activation='relu'),
        MaxPooling2D(2,2),

        Flatten(),

        Dense(512 , activation = 'relu'),
        Dropout(0.5),

        Dense(1,activation='sigmoid')
    ]
)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


# **Compile the model:**

Now we will compile the model and use some metric, loss,optimizer

In [30]:
model.compile(
    optimizer=Adam(learning_rate=0.0001),
    metrics = ['accuracy'],
    loss = 'binary_crossentropy'
)

In [ ]:
model.summary()

# **Train the model:**

In [31]:
history = model.fit(
    train_data,

    epochs = 20,
    validation_data = val_data
)

Epoch 1/20
  9/547 ━━━━━━━━━━━━━━━━━━━━ 17:36 2s/step - accuracy: 0.5603 - loss: 0.6941

KeyboardInterrupt: 

# **Visulize the graph:**

In [ ]:

plt.plot(history.history['accuracy'], label='Training Accuracy', color='blue')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy', color='green')
plt.title('Model ki Accuracy')
plt.xlabel('Epochs (Rounds)')
plt.ylabel('Accuracy (Percentage)')
plt.legend()
plt.show()

plt.plot(history.history['loss'], label='Training Loss', color='red')
plt.plot(history.history['val_loss'], label='Validation Loss', color='orange')
plt.title('Model ka Loss (Galti)')
plt.xlabel('Epochs (Rounds)')
plt.ylabel('Loss')
plt.legend()
plt.show()


# **Evaluation of Model on Test**

In [ ]:

print("Model Final Exam Start...")


test_loss, test_accuracy = model.evaluate(test_data)

print(f"\nTest Accuracy: {test_accuracy * 100:.2f}%")
print(f"Test Evaluate: {test_loss:.4f}")


# **Save trained model:**

In [ ]:
model.save('cats_dogs_model.keras')